# 33 - 拆 Claude Code 的 Skill 机制 + 沙箱建模

> 学习目标：理解 Skill 在 Claude Code 系统中的位置、存放结构、加载流程。在沙箱里建出 `~/.claude/skills/` 的镜像。
> 预备：02-Agent 跑过；理解 sub-agent / MCP / hooks。

Skill 一句话: `~/.claude/skills/<name>/SKILL.md` + 可选附属资源 + 约定 frontmatter。Claude Code 每次会话扫所有 description 按 query 召回。

为避免编辑真实 `~/.claude/`, 所有 Skill 落到 `./_skill_sandbox/home/.claude/skills/`。

In [ ]:
import os, shutil, json
from pathlib import Path
import yaml
print('imported: yaml / Path / json')

## 1. 在沙箱里建 home 镜像

In [ ]:
SBX = Path('./_skill_sandbox').resolve()
if SBX.exists():
    shutil.rmtree(SBX)
SBX.mkdir()
FAKE_HOME = SBX / 'home'
FAKE_HOME.mkdir()
FAKE_SKILLS = FAKE_HOME / '.claude' / 'skills'
FAKE_SKILLS.mkdir(parents=True)
print('sandbox ready at', SBX)

## 2. 写一个 SKILL.md (5 段模板)

In [ ]:
skill_dir = FAKE_SKILLS / 'explain-code'
skill_dir.mkdir()
skill_md = skill_dir / 'SKILL.md'
SKILL_CONTENT = """---
name: explain-code
description: |
  When the user asks to explain a snippet of code, a function, a class, or a section of a file.
  Trigger phrases include explain this code, what does this function do.
  Do NOT trigger for rewriting code, reviewing PRs, or generating new code.
---

# When to use
User wants a clear natural-language explanation of existing code.

# When NOT to use
- User wants to rewrite or fix the code
- User wants a PR-level review
- User wants to generate new code from scratch

# Steps
1. Read the snippet with the Read tool
2. Identify the language and key constructs
3. Write a structured explanation
4. If anything is ambiguous ask the user instead of guessing

# Example
User asks explain this decorator
You read the code and produce a structured explanation.
"""
skill_md.write_text(SKILL_CONTENT, encoding='utf-8')
print(f'created {skill_md} ({skill_md.stat().st_size} bytes)')


In [ ]:
def parse_skill_md(path):
    text = path.read_text(encoding='utf-8')
    if not text.startswith('---'):
        return {'name': '?', 'description': '?', 'body': text}
    end = text.find('\n---', 3)
    if end == -1:
        return {'name': '?', 'description': '?', 'body': text}
    fm = yaml.safe_load(text[3:end])
    body = text[end+4:].strip()
    return dict(fm, body=body)

parsed = parse_skill_md(skill_md)
print('parsed:')
for k, v in parsed.items():
    if isinstance(v, str) and len(v) > 80:
        print(f'  {k}: {v[:80]}...')
    else:
        print(f'  {k}: {v}')

## 3. SkillLoader 多 root 加载

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Skill:
    name: str
    description: str
    body: str
    path: Path
    resources: list = field(default_factory=list)

class SkillLoader:
    def __init__(self, roots):
        self.roots = [Path(r) for r in roots]
        self.skills = {}

    def load(self):
        for root in self.roots:
            if not root.is_dir():
                continue
            for entry in sorted(root.iterdir()):
                if not entry.is_dir():
                    continue
                sm = entry / 'SKILL.md'
                if not sm.is_file():
                    continue
                parsed = parse_skill_md(sm)
                res = sorted([p for p in entry.rglob('*') if p.is_file() and p.name != 'SKILL.md'])
                self.skills[parsed['name']] = Skill(
                    name=parsed['name'],
                    description=parsed['description'],
                    body=parsed['body'],
                    path=sm,
                    resources=res,
                )
        return self.skills

    def list_descriptions(self):
        return [(s.name, s.description) for s in self.skills.values()]

loader = SkillLoader([FAKE_SKILLS])
loader.load()
print(f'loaded {len(loader.skills)} skill(s):')
for name, desc in loader.list_descriptions():
    print(f'  - {name}: {desc[:60].strip()}...')

## 4. 多级加载：项目级 Skill 覆盖用户级

In [ ]:
proj_skills = SBX / 'my_project' / '.claude' / 'skills'
proj_skills.mkdir(parents=True)
proj_skill = proj_skills / 'explain-code'
proj_skill.mkdir()
OVERRIDE_CONTENT = """---
name: explain-code
description: |
  [PROJECT OVERRIDE] Always start with a security audit section before explaining.
  Trigger on any code question in this project.
---

# Project rules
1. Security first: list any obvious vulnerabilities BEFORE explaining the logic
2. Then follow the user-level explain-code template
3. Always link to our internal wiki at the end
"""
(proj_skill / 'SKILL.md').write_text(OVERRIDE_CONTENT, encoding='utf-8')

loader2 = SkillLoader([FAKE_SKILLS, proj_skills])
loader2.load()
print(f'merged {len(loader2.skills)} skill(s):')
for s in loader2.skills.values():
    print(f'  - {s.name}  (from {s.path.relative_to(SBX)})')
    print(f'    desc: {s.description[:60].strip()}...')


In [ ]:
shutil.rmtree(SBX, ignore_errors=True)
print('sandbox cleaned')

## 深入思考

1. **为什么 description 用英文？** 习惯做法 (Anthropic 官方示例)。description 是给 LLM 看的召回依据。
2. **Skill 里能直接调工具吗？** 间接通过「步骤」指引 Claude Code 自己用工具。Skill 本身是 markdown。
3. **Skill 和 prompt template 库有什么本质区别？** Prompt template 是人主动调的字符串插值；Skill 是 Claude 自动按 description 召回。
4. **Skill 名字约定？** kebab-case。目录名 = skill 名。
5. **Skill 能调另一个 Skill 吗？** 间接：Skill A 步骤里写「用 /skill-B」。

改一改: 故意写坏一个 Skill（前 matter 没闭合），看 parse_skill_md 怎么退化。

## 自检

- [ ] 默写 SKILL.md frontmatter 字段
- [ ] 解释项目级 Skill 覆盖用户级
- [ ] Skill vs prompt template 的 3 个差别
- [ ] 默写 SkillLoader 扫描逻辑

下一步: 34_first_skill.ipynb